# Deployed BERT hint API — smoke test (all 4 modes)

Sends one **`POST /predict`** per mode (`en_4`, `en_5`, `hr_4`, `hr_5`) against the live Hugging Face Space.

- **Base URL:** set `WORD_LADDER_API_BASE` or edit the default below. Use the **`.hf.space`** host (not `huggingface.co/spaces/.../path`). Docs: [Swagger UI](https://ldoric-word-ladder-api.hf.space/docs).
- **Pairs:** words must exist in the API’s island word lists (same graphs as `data/islands/*_largest_island.txt`).
- **hr_5 test pair (`stric` → `širom`):** on `words_hr_5.txt`, **`stric` has only one one-letter neighbor: `strip`**. The API will always return `strip` as `best_neighbor` for that `current` — that is expected, not a missing response.
- **Optional:** if the Space is private, set `HUGGING_FACE_HUB_TOKEN` or `HF_TOKEN` for the `Authorization` header.
- **Slow / hang:** increase `WORD_LADDER_API_TIMEOUT` (seconds, default 600) or run the `hr_5` request alone after a cold start.

```bash
pip install requests
```

In [ ]:
import json
import os

import requests

BASE = os.environ.get("WORD_LADDER_API_BASE", "https://ldoric-word-ladder-api.hf.space").rstrip("/")
TOKEN = os.environ.get("HUGGING_FACE_HUB_TOKEN") or os.environ.get("HF_TOKEN")
REQUEST_TIMEOUT = int(os.environ.get("WORD_LADDER_API_TIMEOUT", "600"))

CASES = [
    {"mode": "hr_5", "current": "stric", "target": "širom"},
    {"mode": "en_4", "current": "cold", "target": "warm"},
    {"mode": "en_5", "current": "crane", "target": "flame"},
    {"mode": "hr_4", "current": "leća", "target": "ježi"},
]

headers = {"Content-Type": "application/json", "accept": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

r_health = requests.get(f"{BASE}/health", headers=headers, timeout=REQUEST_TIMEOUT)
print("GET /health", r_health.status_code, r_health.text[:500])
print()

for body in CASES:
    try:
        resp = requests.post(
            f"{BASE}/predict",
            headers=headers,
            json={**body, "full_ranking": False},
            timeout=REQUEST_TIMEOUT,
        )
    except requests.RequestException as e:
        print("POST /predict", body["mode"], "REQUEST FAILED:", repr(e))
        print("---")
        continue
    print("POST /predict", body["mode"], resp.status_code)
    try:
        print(json.dumps(resp.json(), ensure_ascii=False, indent=2))
    except Exception:
        print(resp.text[:1000])
    print("---")

GET /health 200 {"status":"ok","device":"cpu","loaded_modes":["hr_4","hr_5","en_4","en_5"]}

POST /predict en_4 200
{
  "best_neighbor": "cord",
  "predicted_distance": 2.4733359813690186,
  "message": null,
  "all_ranked": null
}
---
POST /predict en_5 200
{
  "best_neighbor": "craze",
  "predicted_distance": 1.1544067859649658,
  "message": null,
  "all_ranked": null
}
---
POST /predict hr_4 200
{
  "best_neighbor": "leći",
  "predicted_distance": 1.98445463180542,
  "message": null,
  "all_ranked": null
}
---
POST /predict hr_5 200
{
  "best_neighbor": "strip",
  "predicted_distance": 6.447764873504639,
  "message": null,
  "all_ranked": null
}
---
